<a href="https://colab.research.google.com/github/nannastahl/CB2330-project/blob/main/Projekt_slump_proteinr%C3%B6relse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import matplotlib.pyplot as plt
steps = 3000
T = 1.0 #Temperatur????????+

A_LCG = 1664525
C_LCG = 1013904223
M_LCG = 2 ** 32

seed = 30

movements = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def lcg(state, a, c, m):
    """The next state of a linear congruential generator: s_{i+1} = (a * s_i + c) mod m."""
    return (a * state + c) % m

def protein_initial_position(seed, n_proteins=150, grid_size=50):
    grid = np.zeros((grid_size, grid_size))
    proteins = []
    state = seed
    while len(proteins) < n_proteins:
        state = lcg(state, A_LCG, C_LCG, M_LCG)
        x = int((state / M_LCG) * grid_size) #Generate random number between 0 and 1 and multiply by grid size to get coordinate (whole number)
        state = lcg(state, A_LCG, C_LCG, M_LCG)
        y = int((state / M_LCG) * grid_size)
        if grid[x, y] == 0: #Only puts protein on a free space
            grid[x, y] = 1
            proteins.append([x, y])
    return grid, proteins, state



def energy_calculator(x, y, grid, theta, grid_size=50):
    energy = 0.0
    for dx, dy in movements:
      neighbour_grid_x = x + dx # Possible x-position
      neighbour_grid_y = y + dy #Possible y-position (neighbouring grid based on movements)
      if 0 <= neighbour_grid_x < grid_size and 0 <= neighbour_grid_y < grid_size:
          if grid[neighbour_grid_x, neighbour_grid_y] == 1:
              energy -= 1        #Om ockuperad, minska energi (aggregat)
    if x < grid_size // 2:     #Om på DOPS sida, minska energi med theta som vi ändrar med ABC MCMC
        energy -= theta
    return energy



def random_walk(seed, grid, proteins, theta, grid_size=50):
    state = seed
    #Random protein from list of proteins
    state = lcg(state, A_LCG, C_LCG, M_LCG)
    protein_idx = int((state / M_LCG) * len(proteins))
    x, y = proteins[protein_idx]

    #Random movement
    state = lcg(state, A_LCG, C_LCG, M_LCG)
    move_idx = int((state / M_LCG) * len(movements)) #Could never be bigger than the possible movements, eg. not bigger than 4
    dx, dy = movements[move_idx]
    new_x = x + dx
    new_y = y + dy

    if not (0 <= new_x < grid_size and 0 <= new_y < grid_size): #Only move inside grid
        return state, grid, proteins
    if grid[new_x, new_y] == 1:
        return state, grid, proteins

    old_energy = energy_calculator(x, y, grid, theta, grid_size)
    grid[x, y] = 0 #Old position is empty


    #Räkna energin efter rörelsen

    new_energy = energy_calculator(new_x, new_y, grid, theta, grid_size)

    delta_E = new_energy - old_energy





